# C9-dimensionality-reduction — Practice p10 — Solution

In [1]:
import numpy as np

SEED = 20260804
rng = np.random.default_rng(SEED)
n = 600
t = np.sort(rng.uniform(-1.5 * np.pi, 1.5 * np.pi, n))
y = rng.uniform(0.0, 6.0, n)
X3 = np.column_stack([np.sin(t), y, np.sign(t) * (np.cos(t) - 1.0)]) \
     + rng.normal(0.0, 0.03, (n, 3))
Xc = X3 - X3.mean(axis=0)
_, _, Vt = np.linalg.svd(Xc, full_matrices=False)
P2 = Xc @ Vt[:2].T
UV = np.column_stack([t, y])

D_orig = np.sqrt(((X3[:, None, :] - X3[None, :, :]) ** 2).sum(axis=2))
D_pca = np.sqrt(((P2[:, None, :] - P2[None, :, :]) ** 2).sum(axis=2))
D_unr = np.sqrt(((UV[:, None, :] - UV[None, :, :]) ** 2).sum(axis=2))
proj_never_grows = bool(np.max(D_pca - D_orig) <= 1e-9)
iu = np.triu_indices(n, 1)
stretch_pca = float(np.max(D_pca[iu] / D_orig[iu]))
stretch_unr = float(np.max(D_unr[iu] / D_orig[iu]))
ends_ratio = float(D_unr[0, 599] / D_orig[0, 599])

print("projection guarantee:", proj_never_grows)
print("worst stretches, PCA / unrolled:", stretch_pca, stretch_unr)
print("end-to-end unrolled ratio:", ends_ratio)

projection guarantee: True
worst stretches, PCA / unrolled: 0.9999999999998912 4.303199004892209
end-to-end unrolled ratio: 2.567134477706411


The principal-component view answers straight-line, global-distance questions honestly in the one-sided sense: its worst stretch is $1.0000$, so it never exaggerates distance.  The unrolled view answers along-ribbon neighborhood questions, but its worst stretch is $4.3032$ and its end-to-end ratio is $2.5671$, so its global ruler is not quantitative.  Because the projection never grows a distance, far apart in its view certifies far apart in the original space, whereas close points may be false neighbors created by discarded directions.

### Answer check

In [2]:
assert D_orig.shape == D_pca.shape == D_unr.shape == (600, 600)
assert proj_never_grows
assert np.max(D_pca - D_orig) <= 1e-9
assert np.isclose(stretch_pca, 1.0, atol=1e-12, rtol=0)
assert np.isclose(stretch_unr, 4.303199004892209, atol=1e-12, rtol=0)
assert np.isclose(ends_ratio, 2.567134477706411, atol=1e-12, rtol=0)